In [3]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    
    is_sus = random.random() < 0.05

    if is_sus:
        amount = round(random.uniform(3000.01, 7000), 2)
        category = 'elektronika'
        hour = random.randint(0,5)
    else:
        amount = round(random.uniform(5.0, 5000.0), 2)
        category = random.choice(kategorie)
        hour = random.randint(6, 24)
    
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': amount,
        'store': random.choice(sklepy),
        'category': category,
        'hour': hour,
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('lab1', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [15]:
%%file consumer.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'lab1',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
        print(message)

Overwriting consumer.py


In [7]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'lab1',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    if message.value['amount'] > 1000:
        print(f'alert {message.value}')

Overwriting consumer_filter.py


In [9]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'lab1',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    transaction = message.value
    if transaction['amount'] > 3000:
        risk_level = 'HIGH'
    elif transaction['amount'] > 1000:
        risk_level = 'MEDIUM'
    else:
        risk_level = 'LOW'
    transaction['risk_level'] = risk_level
    print(transaction)
    

Overwriting consumer_enrich.py


In [14]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'lab1',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0
window_size = 10

try:
    for message in consumer:
        tx = message.value
        store = tx.get('store', 'Unknown')
        amount = tx.get('amount', 0.0)

        store_counts[store] += 1
        total_amount[store] += amount
        msg_count += 1

        if msg_count % window_size == 0:
            print('---------------------------------------------------')
            for sklep, count in store_counts.most_common():
                print(f"Sklep: {sklep}, Liczba transakcji: {count}, Suma: {total_amount[sklep]:.2f} PLN")
            print('---------------------------------------------------')
            store_counts.clear()
            total_amount.clear()
except KeyboardInterrupt:
    print("\nZatrzymano konsumenta.")
finally:
    consumer.close()

Overwriting consumer_count.py


In [20]:
def score_transaction(tx):
    
    score = 0
    rules = []

    R1 = tx.get('amount') > 3000.0
    R2 = tx.get('amount') > 1500.0 and tx['category'] == 'elektronika'
    R3 = tx.get('hour') < 6

    if R1:
        score += 3
        rules.append('R1')
    if R2:
        score += 2
        rules.append('R2')
    if R3:
        score += 2
        rules.append('R3')

    return score, rules

# Test
test_tx = {'tx_id': 'TX999', 'amount': 4500.0, 'category': 'elektronika',
           'timestamp': '2026-04-01T03:15:00', 'hour': 12}
print(score_transaction(test_tx))  # powinno dać score >= 5

(5, ['R1', 'R2'])


In [32]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json

def score_transaction(tx):
    
    score = 0
    rules = []

    R1 = tx.get('amount') > 3000.0
    R2 = tx.get('amount') > 1500.0 and tx['category'] == 'elektronika'
    R3 = tx.get('hour') < 6

    if R1:
        score += 3
        rules.append('R1')
    if R2:
        score += 2
        rules.append('R2')
    if R3:
        score += 2
        rules.append('R3')

    return score, rules


consumer = KafkaConsumer(
    'lab1', 
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

for message in consumer:
    tran = message.value
    score = score_transaction(tran)[0]
    if score >= 3:
        alert_producer.send('alerts', value=f'alert {tran}')
        
alert_producer.flush()
alert_producer.close()

Overwriting scoring_consumer.py


In [33]:
%%file alerts_consumer.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'alerts',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
        print(message.value)

Overwriting alerts_consumer.py
